In [ ]:
import os 
import numpy as np
import cv2
import pandas as pd
from skimage.morphology import skeletonize
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt
import seaborn as sns
import tifffile
from plantcv import plantcv as pcv
from concurrent.futures import ThreadPoolExecutor, as_completed

### --> Modify Those Parameters <--

In [ ]:
whiteBG=False
path="......""
#path=thisdrive
identifyer="tif"
cutoff = 400  # Segment length cutoff
stepsize= 1 #Steps = stepping for angle calculation in pixels

##### Detected Files

In [ ]:
subDirs=["tables","plots"]
for s in subDirs:
    result_path = path+"//results//"
    if not os.path.exists(result_path+s):
        os.makedirs(result_path+s+"//")

files=[]
tmp=os.listdir(path)
for f in tmp:
    if (identifyer in f)==True:  
        files=files+[f]
files

##### Functions

In [ ]:
def processStack(image_path,cutoff):
    # Load image (grayscale)
    #image_path = r"Z:\PilotExperimentsAndTrainings2\AG-Owald_Niccolo_Pampaloni\data\2024.10.28_.tif"
    global imageStack
    imageStack = tifffile.imread(image_path)
        
    def process_timepoint(timepoint):
        """Process a single timepoint, return filtered x, y, t values."""
        image = imageStack[timepoint, :, :]
        
        if image is None:
            raise FileNotFoundError(f"Error: Could not load image at {image_path}")
        
        if(whiteBG==True):
            image=image.max()-image
        
        # Apply Otsu thresholding
        thresh_value = threshold_otsu(image)
        binary = (image > thresh_value).astype(np.uint8) * 255
    
        # Closing function (to fill small gaps)
        def closing(image, iterations=5):
            kernel = np.ones((3, 3), np.uint8)
            closed = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel, iterations=iterations)
            return closed
    
        binary = closing(binary, iterations=5)
    
        # Stronger dilation to close gaps
        for _ in range(20):
            binary = cv2.dilate(binary, np.ones((3, 3), np.uint8), iterations=1)
    
        # Skeletonization
        skeleton = skeletonize(binary // 255) * 255
        skeleton = skeleton.astype(np.uint8)
    
        # Additional closing of gaps
        close_gaps = 4
        for _ in range(close_gaps):
            skeleton = cv2.dilate(skeleton, np.ones((3, 3), np.uint8), iterations=1)
    
        # Re-skeletonize after dilation
        skeleton = skeletonize(skeleton // 255) * 255
        skeleton = skeleton.astype(np.uint8)
    
        # Prune Skeleton
        pcv.params.debug = None
        pruned_skeleton, segmented_img, segment_objects = pcv.morphology.prune(skeleton, size=500)
    
        x_values, y_values, t_values = [], [], []
        
        for obj in segment_objects:
            if len(obj) > cutoff:
                x_values += obj[:, 0, 0].tolist()
                y_values += obj[:, 0, 1].tolist()
                t_values += [timepoint] * len(obj)
    
        return x_values, y_values, t_values
    
    # Run with multithreading
    if __name__ == "__main__":
        num_threads = 5  # Number of threads
        timepoints = range(len(imageStack))
    
        results_dict = {}
    
        with ThreadPoolExecutor(max_workers=num_threads) as executor:
            futures = {executor.submit(process_timepoint, t): t for t in timepoints}
    
            for future in as_completed(futures):
                timepoint = futures[future]
                try:
                    x_vals, y_vals, t_vals = future.result()
                    results_dict[timepoint] = (x_vals, y_vals, t_vals)
                except Exception as e:
                    print(f"Error processing timepoint {timepoint}: {e}")
    
        # Merge results
        all_x, all_y, all_t = [], [], []
        for t in sorted(results_dict.keys()):  # Ensure order is maintained
            x_vals, y_vals, t_vals = results_dict[t]
            all_x.extend(x_vals)
            all_y.extend(y_vals)
            all_t.extend(t_vals)



    
    
    # Create DataFrame
    filteredSkeleton = pd.DataFrame({"x": all_x, "y": all_y, "t": all_t})
    
    
    filteredSkeleton["displacement_X"]=filteredSkeleton.groupby("t")["x"].diff().abs()
    filteredSkeleton["displacement_Y"]=filteredSkeleton.groupby("t")["y"].diff().abs()
    filteredSkeleton["displacement_abs"]=(filteredSkeleton["displacement_X"]**2+filteredSkeleton["displacement_Y"]**2)**0.5
     
    return filteredSkeleton

In [ ]:
def binnedAnalysis(filteredSkeleton,stepsize):
    binnedData=pd.DataFrame()
    # Process each timepoint efficiently
    for t, group in filteredSkeleton.groupby("t"):
        length = len(group)
        binnedData=pd.concat([binnedData,filteredSkeleton.loc[filteredSkeleton.t == t].iloc[::stepsize].reset_index(drop=True)])      
    
    # Compute first derivatives (tangent direction)
    dx = np.gradient(binnedData["x"])
    dy = np.gradient(binnedData["y"])
    angles = np.arctan2(dy, dx)  # Compute angles in radians
    
    # Compute change in angle over time
    binnedData["angle"] = np.degrees(angles)  # Convert to degrees
    binnedData["delta_angle"] = binnedData.groupby("t")["angle"].diff().abs()  # Change in angle over time
    
    return binnedData


In [ ]:
def plotOverlay(filteredSkeleton,result_path,f):
    
    # Convert 'filteredSkeleton' DataFrame columns to NumPy arrays
    x = filteredSkeleton["x"].values
    y = filteredSkeleton["y"].values
    t = filteredSkeleton["t"].values  # Time values for color mapping
    
    # Create a color map based on time values
    cmap = plt.get_cmap("viridis")  # Try 'jet', 'plasma', or 'coolwarm' for variety
    #norm = plt.Normalize(min(t), max(t))
    norm = plt.Normalize(vmin=min(t), vmax=max(t))
    colors = cmap(norm(t))
    
    


    
    # Create a 2D scatter plot
    plt.figure(figsize=(10, 8))
    sc = plt.scatter(x, y, c=t, cmap="viridis", alpha=0.01, s=1)  # Alpha=0.1 for transparency, s=1 for small dots
    
    # Labels and title
    plt.xlabel("X Coordinate")
    plt.ylabel("Y Coordinate")
    plt.title("2D Plot of the Skeleton (Color-coded by Time)")
    
    # Colorbar
    from matplotlib.cm import ScalarMappable
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])  # Required for colorbar
    cbar = plt.colorbar(sm)
    cbar.set_label("Time")
    
    # Show plot
    #plt.show()
    plt.savefig(result_path+"plots//"+f[:-4]+"Overlay.png")
    plt.savefig(result_path+"plots//"+f[:-4]+"Overlay.eps")


In [ ]:
def plotAngles(filteredSkeleton, result_path, f, stepsize):
    """Plot the mean angle change over time with a professional style."""
    
    # Apply Seaborn style
    sns.set_theme(style="whitegrid")
    
    plt.figure(figsize=(12, 6))
    
    # Compute the mean of angle changes per time step
    tmp=binnedAnalysis(filteredSkeleton, stepsize)
    angle_change_avg = tmp.groupby("t")["delta_angle"].mean()


    # Plot with enhanced visuals
    plt.plot(angle_change_avg, marker="o", linestyle="-", color="black", 
             alpha=0.8, linewidth=2, markersize=6, label=r"Mean $\Delta$ Angle")

    # Fill under the curve for clarity
    plt.fill_between(angle_change_avg.index, angle_change_avg, color="black", alpha=0.2)

    # Formatting
    plt.xlabel("Time Step", fontsize=14, fontweight="bold")
    plt.ylabel("Average Angle Change (°)", fontsize=14, fontweight="bold")
    plt.title(f"Mean Angle Change Over Time (Stepsize: {stepsize})", fontsize=16, fontweight="bold")

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # Legend positioned to the right for consistency
   # plt.legend(fontsize=12, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)

    # Grid styling
    plt.grid(True, linestyle="--", alpha=0.5)

    # Adjust layout for a clean appearance
    plt.tight_layout()

    # Save plot
    plt.savefig(result_path + "plots/" + f[:-4] + "Angles.png", dpi=300)
    plt.savefig(result_path + "plots/" + f[:-4] + "Angles.eps", dpi=300)
    tmp.to_csv(result_path+"tables//binnedData_"+f[:-3]+"csv")
    # Show plot
    plt.show()


In [ ]:
def plotAnglesHistogram(filteredSkeleton, result_path, f, stepsize):
    """Plot the histogram of mean angle change over time with a professional style."""
    
    # Apply Seaborn style
    sns.set_theme(style="whitegrid")
    
    plt.figure(figsize=(12, 6))
    
    # Compute the mean of angle changes per time step
    tmp = binnedAnalysis(filteredSkeleton, stepsize)
    angle_change_avg = tmp.groupby("t")["delta_angle"].mean()

    # Plot the histogram of angle changes
    plt.hist(angle_change_avg, bins=30, color="black", alpha=0.7, edgecolor="white", label=r"Histogram of $\Delta$ Angle", orientation='vertical')

    # Formatting
    plt.xlabel("Average Angle Change (°)", fontsize=14, fontweight="bold")
    plt.ylabel("Frequency", fontsize=14, fontweight="bold")
    plt.title(f"Histogram of Mean Angle Change (Stepsize: {stepsize})", fontsize=16, fontweight="bold")

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # Legend positioned to the right for consistency
    #plt.legend(fontsize=12, loc="upper right", frameon=False)

    # Grid styling
    plt.grid(True, linestyle="--", alpha=0.5)

    # Adjust layout for a clean appearance
    plt.tight_layout()

    # Save plot
    plt.savefig(result_path + "plots/" + f[:-4] + "Angles_Histogram.png", dpi=300)
    plt.savefig(result_path + "plots/" + f[:-4] + "Angles_Histogram.eps", dpi=300)
    #tmp.to_csv(result_path + "tables//binnedData_" + f[:-3] + "csv")

    # Show plot
    plt.show()
    
    # Save histogram data to CSV
    hist, bin_edges = np.histogram(angle_change_avg, bins=30)
    hist_df = pd.DataFrame({
        "bin_start": bin_edges[:-1],
        "bin_end": bin_edges[1:],
        "count": hist
    })
    hist_df.to_csv(result_path + "tables/" + f[:-4] + "_Angles_Histogram.csv", index=False)



In [ ]:
def plotDisplacement(filteredSkeleton, result_path, f):
    # Apply a scientific style
    sns.set_style("ticks")
    plt.rcParams.update({
        "font.family": "serif",  # Use a scientific font
        "axes.labelsize": 14,    # Larger axis labels
        "axes.titlesize": 16,    # Larger title
        "legend.fontsize": 12,   # Legend font size
        "xtick.labelsize": 12,   # Tick label size
        "ytick.labelsize": 12,
        "figure.dpi": 300,       # High-resolution for publications
        "axes.grid": True,       # Keep a subtle grid
        "grid.alpha": 0.3,       # Make grid less intrusive
        "lines.linewidth": 2,    # Thicker lines for visibility
        "savefig.format": "png"  # Save plots in high-quality format
    })
    
    # Calculate the mean of absolute values of the differences for each time step
    x_change_mean = filteredSkeleton.groupby("t")["displacement_X"].mean()
    y_change_mean = filteredSkeleton.groupby("t")["displacement_Y"].mean()
    abs_change_mean = filteredSkeleton.groupby("t")["displacement_abs"].mean()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))  # Scientific papers often use this ratio
    
    # Plot mean changes in X and Y
    ax.plot(x_change_mean, label=r"Mean $\Delta X$", color="royalblue", alpha=0.9)
    ax.plot(y_change_mean, label=r"Mean $\Delta Y$", color="crimson", alpha=0.9)
    ax.plot(abs_change_mean, label=r"Mean $\Delta Total$", color="black", alpha=0.9)
    
    
    # Fill under curves for better visualization
    ax.fill_between(x_change_mean.index, x_change_mean, color="royalblue", alpha=0.2)
    ax.fill_between(y_change_mean.index, y_change_mean, color="crimson", alpha=0.2)
    ax.fill_between(abs_change_mean.index, abs_change_mean, color="black", alpha=0.2)
    
    # Labels and formatting with LaTeX-style math text
    ax.set_xlabel("Time Step", fontsize=14)
    ax.set_ylabel("Mean Absolute Change", fontsize=14)
    ax.set_title("Temporal Evolution of Movement Activity", fontsize=16)
    
    # Fine-tune axes
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="in", length=6)
    
    # Add legend and place it to the right
    ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
    
    # Adjust layout
    plt.tight_layout()
    
    # Save and show
    
    plt.savefig(result_path + "plots/" + f[:-4] + "Displacement.png", dpi=300, bbox_inches="tight")
    plt.savefig(result_path + "plots/" + f[:-4] + "Displacement.eps", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
def plotDisplacementHistogram(filteredSkeleton, result_path, f):

    # Apply a scientific style
    sns.set_style("ticks")
    plt.rcParams.update({
        "font.family": "serif",  # Use a scientific font
        "axes.labelsize": 14,    # Larger axis labels
        "axes.titlesize": 16,    # Larger title
        "legend.fontsize": 12,   # Legend font size
        "xtick.labelsize": 12,   # Tick label size
        "ytick.labelsize": 12,
        "figure.dpi": 300,       # High-resolution for publications
        "axes.grid": True,       # Keep a subtle grid
        "grid.alpha": 0.3,       # Make grid less intrusive
        "lines.linewidth": 2,    # Thicker lines for visibility
        "savefig.format": "png"  # Save plots in high-quality format
    })
    
    # Calculate the mean of absolute values of the differences for each time step
    x_change_mean = filteredSkeleton.groupby("t")["displacement_X"].mean()
    y_change_mean = filteredSkeleton.groupby("t")["displacement_Y"].mean()
    abs_change_mean = filteredSkeleton.groupby("t")["displacement_abs"].mean()
    
    # Create figure for the histogram
    fig, ax = plt.subplots(figsize=(12, 6))  # Scientific papers often use this ratio
    
    # Plot histogram for x_change_mean and y_change_mean
    ax.hist(x_change_mean, bins=40, alpha=0.6, label=r"Mean $\Delta X$ Displacement", color="royalblue", edgecolor="black", range=(0, 2))
    ax.hist(y_change_mean, bins=40, alpha=0.6, label=r"Mean $\Delta Y$ Displacement", color="crimson", edgecolor="black", range=(0, 2))
    ax.hist(abs_change_mean, bins=40, alpha=0.6, label=r"Mean $\Delta Total$ Displacement", color="black", edgecolor="black", range=(0, 2))
    
    # Labels and formatting with LaTeX-style math text
    ax.set_xlabel("Displacement", fontsize=14)
    ax.set_ylabel("Frequency", fontsize=14)
    ax.set_title("Histogram of Mean Displacements (X and Y)", fontsize=16)
    
    # Fine-tune axes
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="in", length=6)
    
    # Add legend and place it to the right
    ax.legend(frameon=False, loc="upper right")
    
    # Adjust layout
    plt.tight_layout()
    
    # Save and show the plot
    plt.savefig(result_path + "plots/" + f[:-4] + "Displacement_Histogram.png", dpi=300, bbox_inches="tight")
    plt.savefig(result_path + "plots/" + f[:-4] + "Displacement_Histogram.eps", dpi=300, bbox_inches="tight")
    plt.show()

##### Batch Processing

In [ ]:
for f in files:
    image_path=path+"/"+f
    filteredSkeleton=processStack(image_path,cutoff)
    filteredSkeleton.to_csv(result_path+"tables//backbone_"+f[:-3]+"csv")

##### Batch Plotting

In [ ]:
for f in files:
    image_path=path+"/"+f
    result_path=path+"/results/"
    filteredSkeleton=pd.read_csv(result_path+"tables//backbone_"+f[:-3]+"csv",index_col=0)
    binnedAnalysis(filteredSkeleton,stepsize)
    filteredSkeleton.to_csv(result_path+"tables//backbone_"+f[:-3]+"csv")
    plotOverlay(filteredSkeleton,result_path,f)
    plotAngles(filteredSkeleton, result_path, f, stepsize)
    plotAnglesHistogram(filteredSkeleton, result_path, f, stepsize)
    plotDisplacement(filteredSkeleton, result_path, f)
    plotDisplacementHistogram(filteredSkeleton, result_path, f)